# 01 Error analysis: false alerts and missed rises

**Data:** chosen in `common.py` (`USE_FAKE_DATA`). Chart titles say whether it is real or fake.

Every forecast lands in one of four boxes:

| | Actually rose | Didn't rise |
| --- | --- | --- |
| **Model said rising** | Correct alert | False alert (wasted stock) |
| **Model said not rising** | Missed rise (lost sales) | Correct quiet |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from common import load_predictions, RISING_THRESHOLD, DATA_LABEL

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

df = load_predictions()

# Put every forecast in one of the four boxes.
said_rising = df["predicted_growth"] > RISING_THRESHOLD
rose = df["actual_growth"] > RISING_THRESHOLD
df["outcome"] = "Correct quiet"
df.loc[said_rising & rose, "outcome"] = "Correct alert"
df.loc[said_rising & ~rose, "outcome"] = "False alert"
df.loc[~said_rising & rose, "outcome"] = "Missed rise"

# The main model and horizon for the detailed questions below.
MODEL, HORIZON = "lightgbm", 4
main = df[(df["model"] == MODEL) & (df["horizon"] == HORIZON)].copy()
print(f"{len(main):,} forecasts for {MODEL}, {HORIZON} weeks ahead")

## 1. The four boxes for the main model

Counts and shares of all forecasts.

In [ ]:
boxes = main["outcome"].value_counts().reindex(["Correct alert", "False alert", "Missed rise", "Correct quiet"])
pd.DataFrame({"forecasts": boxes, "share (%)": (100 * boxes / boxes.sum()).round(1)})

## 2. Precision and recall for every model

- **Precision** = correct alerts ÷ all alerts. How often an alert is right.
- **Recall** = correct alerts ÷ all real rises. How many real rises were caught.

A model that never says "rising" (like the 4-week average, `avg_4w`) has no precision: it makes no alerts at all.

In [ ]:
def alert_scores(group):
    counts = group["outcome"].value_counts()
    correct = counts.get("Correct alert", 0)
    false = counts.get("False alert", 0)
    missed = counts.get("Missed rise", 0)
    alerts = correct + false
    return pd.Series({
        "alerts": alerts,
        "false alerts": false,
        "missed rises": missed,
        "precision (%)": round(100 * correct / alerts, 1) if alerts else None,
        "recall (%)": round(100 * correct / (correct + missed), 1),
    })

scores = df.groupby(["horizon", "model"])[["outcome"]].apply(alert_scores)
counts = ["alerts", "false alerts", "missed rises"]
scores[counts] = scores[counts].astype(int)
scores

## 3. When do false alerts happen?

Share of forecasts that were false alerts, per month the forecast was made. The shaded area is the spring 2026 spike period (March–May).

In [ ]:
main["month"] = main["origin"].dt.to_period("M").dt.to_timestamp()
monthly = main.groupby("month")["outcome"].apply(lambda s: 100 * (s == "False alert").mean())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly.index, monthly.values, width=26, align="edge")  # each bar fills its month
ax.axvspan(pd.Timestamp("2026-03-01"), pd.Timestamp("2026-05-31"), color="grey", alpha=0.2, label="Spring 2026 spike")
ax.set_title(f"False alerts per month: {MODEL}, {HORIZON} weeks ahead ({DATA_LABEL})")
ax.set_xlabel("Month the forecast was made")
ax.set_ylabel("False alerts (% of forecasts)")
ax.legend()
fig.savefig(FIGURES / "01_false_alerts_by_month.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Why: does the model chase recent jumps?

**Recent jump** = last 4-week average ÷ the 4-week average one month earlier. Above 1 means the style has just gone up.

If false alerts have much bigger recent jumps than correct alerts, the model is mistaking short bursts for lasting trends.

In [ ]:
jumps = (main.dropna(subset=["recent_jump"])
             .groupby("outcome")["recent_jump"]
             .median()
             .round(2)
             .reindex(["Correct alert", "False alert", "Missed rise", "Correct quiet"]))
pd.DataFrame({"median recent jump": jumps})

## 5. Where: mistakes by category

Share of forecasts in each category that were false alerts or missed rises, sorted by false alerts.

In [ ]:
by_category = (main.groupby("category")["outcome"]
                   .value_counts(normalize=True)
                   .unstack(fill_value=0)
                   .mul(100)
                   .round(1))
by_category[["False alert", "Missed rise"]].sort_values("False alert", ascending=False)

## 6. Which styles are missed most often?

Styles with the most missed rises. New styles with little history are the usual suspects.

In [ ]:
missed = main[main["outcome"] == "Missed rise"]["item"].value_counts().head(10)
pd.DataFrame({"missed rises": missed})

## What this shows

Real data, LightGBM, 137 styles, 4 weeks ahead, threshold 1.2, test period January 2025 – September 2026.

1. **Precision and recall vs. the baselines:** LightGBM's alerts were right 39.6% of the time
   (last value: 61.1%) and caught 28.3% of real rises (last value: 24.8%). It made far more alerts
   (1,723 vs. 976), and more than half were false. At 8 weeks ahead it had the best recall of all
   models (50.0%, next best 29.1%), but lower precision (58.4% vs. 68–69% for Ridge, Lasso and
   seasonal naive).
2. **When false alerts happen:** 81% of all false alerts (840 of 1,041) came from forecasts made in
   May–July 2026, only 15% of all forecasts. In June 2026, 78% of styles were flagged as rising and
   1% rose. Outside those three months, precision was 76.1% (Ridge, the next best: 68.6%; last value: 62.5%).
3. **Why:** after the spring 2026 spike ended, searches fell by about a third (median recent jump
   0.67), and the model predicted a rebound of about 40%. In 2017–2024, sharp drops were usually
   temporary dips, so the model learned to expect recovery. This drop was a return to normal after a
   market-wide spike the model had never seen in training. Outside May–July 2026, false alerts mostly
   followed recent jumps (median 1.33), so the model sometimes chased short bursts.
4. **Where:** false alert rates were similar across categories (7.5% to 9.7%), so mistakes depend on
   timing more than on category. The most-missed rises were established styles, especially sneakers
   (white, chunky and dad sneakers, 22–24 each) and plaid (23).